01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
.appName("IcebergWithSpark") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
.config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
.config("spark.sql.default.catalog", "hadoop_catalog") \
.getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,946 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,931 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Get:14 http://security.ubunt

# 02. Snapshots e TimeTravel

In [10]:
from datetime import datetime, timedelta

In [11]:
# Criamos a tabela vendas usando Iceberg
spark.sql("""
    CREATE TABLE IF NOT EXISTS hadoop_catalog.default.vendas (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")


DataFrame[]

In [12]:
# Dados Iniciais
data_initial = [
    (1, "Produto A", 10, 15.5, "2023-11-01"),
    (2, "Produto B", 5, 22.0, "2023-11-02"),
    (3, "Produto C", 8, 30.0, "2023-11-03")
]

columns = ["id", "produto", "quantidade", "preco", "data_venda"]

df_initial = spark.createDataFrame(data_initial, columns)
df_initial = df_initial.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

# Gravamos os dados
df_initial.writeTo("hadoop_catalog.default.vendas").append()

In [13]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
+---+---------+----------+-----+----------+



In [14]:
# Lista de snapshots atuais da tabela vendas
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4993504210518577532|2026-03-27 20:48:29.901|append   |
+-------------------+-----------------------+---------+



In [15]:
# Incluimos mais dados
data_additional = [
    (4, "Produto D", 12, 25.0, "2023-11-04"),
    (5, "Produto E", 7, 18.5, "2023-11-05")
]

df_additional = spark.createDataFrame(data_additional, columns)
df_additional = df_additional.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))
df_additional.writeTo("hadoop_catalog.default.vendas").append()

In [16]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [17]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4993504210518577532|2026-03-27 20:48:29.901|append   |
|8995527185609808990|2026-03-27 20:49:55.903|append   |
+-------------------+-----------------------+---------+



In [18]:
# Atualização de dados
spark.sql("""
    UPDATE hadoop_catalog.default.vendas
    SET preco = 16.0
    WHERE id = 1
""")

DataFrame[]

In [19]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas  order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  2|Produto B|         5| 22.0|2023-11-02|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [20]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4993504210518577532|2026-03-27 20:48:29.901|append   |
|8995527185609808990|2026-03-27 20:49:55.903|append   |
|4445829671220638815|2026-03-27 20:50:34.605|overwrite|
+-------------------+-----------------------+---------+



In [21]:
# Excluímos dados
spark.sql("""
    DELETE FROM hadoop_catalog.default.vendas
    WHERE id = 2
""")

DataFrame[]

In [22]:
# Visualização dos dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas  order by id").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 16.0|2023-11-01|
|  3|Produto C|         8| 30.0|2023-11-03|
|  4|Produto D|        12| 25.0|2023-11-04|
|  5|Produto E|         7| 18.5|2023-11-05|
+---+---------+----------+-----+----------+



In [23]:
# List os snapshots novamente
snapshots_df = spark.sql("SELECT * FROM hadoop_catalog.default.vendas.snapshots")
snapshots_df.select("snapshot_id", "committed_at", "operation").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4993504210518577532|2026-03-27 20:48:29.901|append   |
|8995527185609808990|2026-03-27 20:49:55.903|append   |
|4445829671220638815|2026-03-27 20:50:34.605|overwrite|
|4137395614141185503|2026-03-27 20:51:51.396|overwrite|
+-------------------+-----------------------+---------+

